In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType, DoubleType

catalog_name = 'ecommerce'

In [0]:
df_silver_customers = spark.table(f"{catalog_name}.bronze.brz_customers")

df_silver_customers.printSchema()

In [0]:
row_count, column_count = df_silver_customers.count(), len(df_silver_customers.columns)

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

In [0]:
df_silver_customers.show(20)

## 1. customer_id

In [0]:
# 1.1 transformation: slv_customers -> customer_id

# None - super clean; most likely system-generated
df_silver_customers = df_silver_customers.dropDuplicates(["customer_id"]).dropna(subset=["customer_id"])


In [0]:
# 1.2 validation: slv_customers -> customer_id

df_silver_customers.select("customer_id").filter(
    F.col("customer_id").isNull() |
    ~F.col("customer_id").like("CUST%") |
    (F.length(F.col("customer_id")) != 17)
).show()

#duplicates
df_silver_customers.groupBy("customer_id").count() \
    .filter(F.col("count") > 1).show()


## 2. phone

In [0]:
# 2.1 transformation: slv_customers -> phone

df_silver_customers = df_silver_customers.withColumn(
    "phone",
    F.regexp_replace(F.col("phone"), "[^0-9]", ""))

df_silver_customers = df_silver_customers.withColumn(
    "is_phone_valid",
    F.when(F.col("phone").isNull(), F.lit(None))
    .when(
        (F.length(F.col("phone")) >= 10) &
        (F.length(F.col("phone")) <= 15) &
        F.col("phone").rlike("^[0-9]"), True
    )
    .otherwise(False)
)

In [0]:
# 2.2 validation: slv_customers -> phone

invalid_phones = df_silver_customers.filter(
    F.col("phone").isNotNull() &
    (F.length(F.col("phone")) < 10) | (F.length(F.col("phone")) > 15)
)

invalid_phones.show(10)


## 3. country_code

## 4. country

## 5. state